# Generate RoB2 Q&A Dataset

Uses an LLM to generate factual Q&A pairs from the scraped RoB2 content.

**Output:** `evaluation_details/evaluation_rob2/full_dataset_rob2.csv`  
**Schema:** `id, source_url, domain, difficulty, question, reference_answer`

Target: ~100 pairs across 5 domains × 3 difficulty levels (Basic / Intermediate / Advanced).

In [ ]:
import os, json, warnings, re
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import pandas as pd
from dotenv import load_dotenv
load_dotenv()

from rag_core import fetch_visible_text, build_llm

## 1. Scrape RoB2 Pages (raw text per domain)

In [ ]:
DOMAIN_URLS = {
    "Overview": "https://www.riskofbias.info/welcome/rob-2-0-tool",
    "Domain 1 - Randomization": "https://www.riskofbias.info/welcome/rob-2-0-tool/rob-2-domain-1-bias-arising-from-the-randomization-process",
    "Domain 2 - Deviations": "https://www.riskofbias.info/welcome/rob-2-0-tool/rob-2-domain-2-bias-due-to-deviations-from-intended-interventions",
    "Domain 3 - Missing Data": "https://www.riskofbias.info/welcome/rob-2-0-tool/rob-2-domain-3-bias-due-to-missing-outcome-data",
    "Domain 4 - Outcome Measurement": "https://www.riskofbias.info/welcome/rob-2-0-tool/rob-2-domain-4-bias-in-measurement-of-the-outcome",
    "Domain 5 - Reported Results": "https://www.riskofbias.info/welcome/rob-2-0-tool/rob-2-domain-5-bias-in-selection-of-the-reported-result",
    "Overall Judgment": "https://www.riskofbias.info/welcome/rob-2-0-tool/rob-2-overall-risk-of-bias-judgement",
}

domain_texts = {}
for domain, url in DOMAIN_URLS.items():
    print(f"Scraping: {domain}")
    try:
        text = fetch_visible_text(url, wait_time=6)
        domain_texts[domain] = {"url": url, "text": text}
        print(f"  -> {len(text)} chars")
    except Exception as e:
        print(f"  WARNING: {e}")
        domain_texts[domain] = {"url": url, "text": ""}

## 2. Generate Q&A Pairs with LLM

In [ ]:
# Use a capable model for generation (not necessarily the same as the chatbot model)
gen_llm = build_llm(model_env_var="MODEL_NAME")

GENERATION_PROMPT = """\
You are a medical methodology expert. Given the following text from the RoB2 (Risk of Bias 2.0) tool,
generate exactly {n} factual question-answer pairs at {difficulty} difficulty level.

Rules:
- Each answer must be directly answerable from the provided text (no external knowledge).
- Basic: single-fact recall questions (definitions, yes/no with explanation).
- Intermediate: questions requiring understanding of relationships between concepts.
- Advanced: scenario-based questions requiring multi-step reasoning.
- Questions should be diverse — do not repeat the same concept.
- Do NOT include questions that are already answered by another pair in the list.

Return ONLY a valid JSON array of objects, each with keys "question" and "answer".
No markdown, no explanation, just the JSON array.

Text:
{text}
"""

def generate_qa_pairs(text: str, domain: str, url: str, difficulty: str, n: int = 5) -> list[dict]:
    if len(text) < 200:
        print(f"  Skipping {domain}/{difficulty} — text too short ({len(text)} chars)")
        return []

    # Truncate to avoid token limits
    truncated = text[:6000]
    prompt = GENERATION_PROMPT.format(n=n, difficulty=difficulty, text=truncated)

    try:
        response = gen_llm.invoke(prompt)
        content = response.content if hasattr(response, 'content') else str(response)

        # Strip markdown code fences if present
        content = re.sub(r'^```(?:json)?\n?', '', content.strip())
        content = re.sub(r'\n?```$', '', content.strip())

        pairs = json.loads(content)
        for p in pairs:
            p["domain"] = domain
            p["source_url"] = url
            p["difficulty"] = difficulty
        return pairs
    except Exception as e:
        print(f"  ERROR generating {domain}/{difficulty}: {e}")
        return []

In [ ]:
# Generate: 5 Basic + 5 Intermediate + 4 Advanced per domain = ~14 per domain × 7 domains ≈ 98 pairs
N_PER_LEVEL = {"Basic": 5, "Intermediate": 5, "Advanced": 4}

all_pairs = []
counter = 1

for domain, info in domain_texts.items():
    print(f"\nGenerating for: {domain}")
    for difficulty, n in N_PER_LEVEL.items():
        pairs = generate_qa_pairs(info["text"], domain, info["url"], difficulty, n=n)
        for p in pairs:
            p["id"] = f"R{counter:04d}"
            counter += 1
        all_pairs.extend(pairs)
        print(f"  {difficulty}: {len(pairs)} pairs generated")

print(f"\nTotal Q&A pairs: {len(all_pairs)}")

## 3. Review & Save

In [ ]:
df = pd.DataFrame(all_pairs)[["id", "source_url", "domain", "difficulty", "question", "answer"]]
df.columns = ["id", "source_url", "domain", "difficulty", "question", "reference_answer"]

print(f"Shape: {df.shape}")
print(f"\nDomain distribution:\n{df['domain'].value_counts()}")
print(f"\nDifficulty distribution:\n{df['difficulty'].value_counts()}")
df.head(5)

In [ ]:
out_dir = "evaluation_details/evaluation_rob2"
os.makedirs(out_dir, exist_ok=True)
out_path = f"{out_dir}/full_dataset_rob2.csv"
df.to_csv(out_path, index=False)
print(f"Saved {len(df)} Q&A pairs to {out_path}")

## 4. Manual Review Sample

In [ ]:
# Print a random sample of 10 for manual review
sample = df.sample(min(10, len(df)), random_state=42)
for _, row in sample.iterrows():
    print(f"[{row['id']}] [{row['domain']}] [{row['difficulty']}]")
    print(f"  Q: {row['question']}")
    print(f"  A: {row['reference_answer'][:200]}")
    print()